# RAG-Based Company Document Visualisation
### AcmeTech Solutions — Synthetic Knowledge Base

| Step | What happens |
|---|---|
| 2 | Install libraries |
| 3 | Load 6 knowledge base files |
| 4 | Split into 90 chunks |
| 4b | Pre-embedding diagnostics |
| 5 | Generate 384-dim embeddings |
| 6 | Store in ChromaDB |
| 7 | Test retrieval |
| 8 | UMAP 2D & 3D reduction |
| 9 | Build enriched plot DataFrame |
| 10 | Interactive cluster plots (2D + 3D) |
| 11 | Save PNG + HTML exports |
| 12 | Advanced visualisation |
| 13 | RAG query with Google Gemini ✅ |

---
## Step 2: Install Libraries

In [ ]:
!pip install -q \
    langchain langchain-community langchain-google-genai \
    faiss-cpu sentence-transformers chromadb \
    plotly umap-learn pandas kaleido

---
## Step 3: Load Documents

In [ ]:
from langchain.schema import Document

file_category_map = {
    "employee_directory.txt":        "employee",
    "hr_policies.txt":               "hr",
    "finance_tax.txt":               "finance",
    "engineering_documentation.txt": "engineering",
    "customer_support_kb.txt":       "support",
    "product_management.txt":        "product",
}
docs = []
for filename, category in file_category_map.items():
    with open(filename, "r", encoding="utf-8") as f:
        content = f.read()
    docs.append(Document(page_content=content,
                         metadata={"category": category, "source": filename}))
print(f"Loaded {len(docs)} files")
for doc in docs:
    print(f"  [{doc.metadata['category']:>12}]  {doc.metadata['source']}  ({len(doc.page_content.split()):,} words)")

---
## Step 4: Chunk Documents

In [ ]:
from collections import Counter

SEPARATOR = "==============================="
chunks = []
for doc in docs:
    for raw in doc.page_content.split(SEPARATOR):
        text = raw.strip()
        if len(text) < 100:
            continue
        chunks.append(Document(page_content=text, metadata=doc.metadata.copy()))

counts = Counter(c.metadata["category"] for c in chunks)
print(f"Total chunks: {len(chunks)}\n")
for cat, n in sorted(counts.items()):
    print(f"  {cat:>12} : {n} chunks")

---
## Step 4b: Visualise Chunks

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_chunks = pd.DataFrame([{
    "category":   c.metadata["category"],
    "source":     c.metadata["source"],
    "word_count": len(c.page_content.split()),
    "preview":    c.page_content[:120].replace("\n", " "),
} for c in chunks])

CATEGORY_COLORS = {
    "employee":    "#636EFA",
    "hr":          "#EF553B",
    "finance":     "#00CC96",
    "engineering": "#AB63FA",
    "support":     "#FFA15A",
    "product":     "#19D3F3",
}
print(df_chunks.groupby("category")["word_count"]
    .agg(chunks="count", min_words="min", max_words="max", avg_words="mean")
    .reset_index().to_string(index=False))

In [ ]:
count_df = df_chunks.groupby("category").size().reset_index(name="chunk_count")
px.bar(count_df, x="category", y="chunk_count", color="category",
    color_discrete_map=CATEGORY_COLORS, text="chunk_count",
    title="📊 Chunk Count per Category").update_traces(
    textposition="outside").update_layout(showlegend=False,
    plot_bgcolor="white", title_font_size=18).show()

In [ ]:
px.box(df_chunks, x="category", y="word_count", color="category",
    color_discrete_map=CATEGORY_COLORS, points="all",
    title="📦 Word Count Distribution per Category").update_layout(
    showlegend=False, plot_bgcolor="white", title_font_size=18).show()

---
## Step 5: Generate Embeddings

**Expected shape: `(90, 384)`**

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
texts      = [chunk.page_content for chunk in chunks]
embeddings = embedding_model.encode(texts, show_progress_bar=True, batch_size=32)

print(f"embeddings.shape : {embeddings.shape}")
assert embeddings.shape == (len(chunks), 384)
print("✅ Shape check passed: (90, 384)")

---
## Step 6: Store in ChromaDB

> ⚠️ Use `langchain_community.embeddings`  
> ℹ️ `db.persist()` not needed in chromadb ≥ 0.4

In [ ]:
import os, shutil
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

PERSIST_DIR = "./vector_db"
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

embedding_fn = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}, encode_kwargs={"normalize_embeddings": True})

db = Chroma.from_documents(documents=chunks, embedding=embedding_fn,
                           persist_directory=PERSIST_DIR)
print(f"✅ ChromaDB — {db._collection.count()} documents stored")

---
## Step 7: Test Retrieval

| Score | Meaning |
|---|---|
| `< 0.30` | 🟢 Strong |
| `0.30–0.60` | 🟡 Good |
| `> 0.60` | 🔴 Weak |

In [ ]:
def run_query(query, k=3):
    print(f"\nQUERY: {query}")
    for rank, (doc, score) in enumerate(db.similarity_search_with_score(query, k=k), 1):
        icon = "🟢" if score < 0.30 else "🟡" if score < 0.60 else "🔴"
        print(f"  #{rank} {icon} [{doc.metadata['category']}] score={score:.4f}")
        print(f"     {doc.page_content[:200].replace(chr(10),' ')}...")

In [ ]:
run_query("How many maternity leave days are allowed?")
run_query("How are Kubernetes deployments configured?")
run_query("How is the annual bonus calculated?")
run_query("How do I reset my account password?")
run_query("What is on the product roadmap?")
run_query("Who manages the engineering team?")

---
## Step 8: UMAP Reduction (384 → 2D & 3D)

In [ ]:
import numpy as np
from umap import UMAP

UMAP_PARAMS = dict(n_neighbors=10, min_dist=0.1, metric="cosine", random_state=42)

print("Running UMAP 384 → 2D ...")
coords_2d = UMAP(n_components=2, **UMAP_PARAMS).fit_transform(embeddings)
print(f"  shape: {coords_2d.shape}")

print("Running UMAP 384 → 3D ...")
coords_3d = UMAP(n_components=3, **UMAP_PARAMS).fit_transform(embeddings)
print(f"  shape: {coords_3d.shape}")

---
## Step 9: Build Enriched DataFrame

In [ ]:
import re

def parse_doc_id(text):
    m = re.search(r"(EMP|HR|FIN|ENG|SUP|PM)-\d+", text)
    return m.group(0) if m else "UNKNOWN"

def parse_title(text):
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith("title:"):
            return line.split(":", 1)[-1].strip()
    for line in text.splitlines():
        line = line.strip()
        if line and not line.startswith("Document") and len(line) > 8:
            return line[:80]
    return "Untitled"

rows = []
for i, chunk in enumerate(chunks):
    text = chunk.page_content.strip()
    rows.append({
        "x":  coords_2d[i, 0], "y":  coords_2d[i, 1],
        "x3": coords_3d[i, 0], "y3": coords_3d[i, 1], "z3": coords_3d[i, 2],
        "category":   chunk.metadata["category"],
        "source":     chunk.metadata["source"],
        "doc_id":     parse_doc_id(text),
        "title":      parse_title(text),
        "word_count": len(text.split()),
        "preview":    text[:120].replace("\n", " "),
    })

df = pd.DataFrame(rows)
assert len(df) == 90 and df["category"].nunique() == 6
print(f"✅ df.shape: {df.shape}")
df[["doc_id", "category", "title", "word_count"]].head(8)

---
## Step 10: Interactive Plots — 2D & 3D

In [ ]:
DARK_BG = "#0f0f1a"
LEGEND_BG = "rgba(255,255,255,0.08)"
LEGEND_BC = "rgba(255,255,255,0.20)"

def dark_layout(fig, is_3d=False):
    axis = dict(showgrid=False, zeroline=False, showticklabels=False, title="")
    base = dict(paper_bgcolor=DARK_BG, font_color="white", title_font_size=19,
                legend=dict(title="Category", bgcolor=LEGEND_BG,
                            bordercolor=LEGEND_BC, borderwidth=1))
    if is_3d:
        base["scene"] = dict(bgcolor=DARK_BG,
            xaxis={**axis, "backgroundcolor": DARK_BG},
            yaxis={**axis, "backgroundcolor": DARK_BG},
            zaxis={**axis, "backgroundcolor": DARK_BG})
    else:
        base["plot_bgcolor"] = DARK_BG
        base["xaxis"] = axis
        base["yaxis"] = axis
    return fig.update_layout(**base)

In [ ]:
fig1 = px.scatter(df, x="x", y="y", color="category",
    color_discrete_map=CATEGORY_COLORS, hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "preview": True,
                "x": False, "y": False, "category": False},
    title="🗺️  Vector Embedding Clusters — UMAP 2D", width=1050, height=680)
fig1.update_traces(marker=dict(size=11, opacity=0.88, line=dict(width=0.6, color="white")))
dark_layout(fig1).show()

fig2 = px.scatter(df, x="x", y="y", color="category",
    color_discrete_map=CATEGORY_COLORS, size="word_count", size_max=22,
    hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "preview": True,
                "x": False, "y": False, "category": False},
    title="🏷️  UMAP 2D — Labels + Point Size = Word Count", width=1050, height=680)
fig2.update_traces(marker=dict(opacity=0.82, line=dict(width=0.5, color="white")))
for cat, color in CATEGORY_COLORS.items():
    sub = df[df["category"] == cat]
    fig2.add_annotation(x=sub["x"].mean(), y=sub["y"].mean(),
        text=f"<b>{cat.upper()}</b>", showarrow=False,
        font=dict(size=14, color=color), bgcolor="rgba(15,15,26,0.70)", borderpad=5)
dark_layout(fig2).show()

In [ ]:
fig3 = px.scatter_3d(df, x="x3", y="y3", z="z3", color="category",
    color_discrete_map=CATEGORY_COLORS, hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "preview": True,
                "x3": False, "y3": False, "z3": False, "category": False},
    title="🌐  Vector Embedding Clusters — UMAP 3D", width=1050, height=780)
fig3.update_traces(marker=dict(size=6, opacity=0.90, line=dict(width=0.3, color="white")))
dark_layout(fig3, is_3d=True)
fig3.update_layout(scene_camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)))
fig3.show()

fig4 = px.scatter_3d(df, x="x3", y="y3", z="z3", color="category",
    color_discrete_map=CATEGORY_COLORS, size="word_count", size_max=14,
    hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "category": True,
                "preview": True, "x3": False, "y3": False, "z3": False},
    title="🔍  UMAP 3D — Point Size = Word Count", width=1050, height=780)
fig4.update_traces(marker=dict(opacity=0.85, line=dict(width=0.2, color="white")))
dark_layout(fig4, is_3d=True)
fig4.update_layout(scene_camera=dict(eye=dict(x=1.8, y=0.8, z=0.6)))
fig4.show()

---
## Step 11: Save PNG + HTML

| Format | Use | Interactive? |
|---|---|---|
| PNG `scale=2` | Slides, emails | ❌ |
| HTML `cdn` | Browser / stakeholder share | ✅ |

In [ ]:
figures = {"umap_2d_clusters": fig1, "umap_2d_labelled": fig2,
           "umap_3d_clusters": fig3, "umap_3d_sized": fig4}

print("Saving PNG images ...")
for name, fig in figures.items():
    try:
        fig.write_image(f"{name}.png", width=1200, height=800, scale=2)
        print(f"  ✅  {name}.png  ({os.path.getsize(name+'.png')//1024} KB)")
    except Exception as e:
        print(f"  ⚠️  {name}.png failed: {e}")

print("\nSaving HTML files ...")
for name, fig in figures.items():
    fig.write_html(f"{name}.html", include_plotlyjs="cdn")
    print(f"  ✅  {name}.html  ({os.path.getsize(name+'.html')//1024} KB)")

In [ ]:
# Colab only — downloads files to your laptop
try:
    from google.colab import files
    for name in figures:
        files.download(f"{name}.png")
        files.download(f"{name}.html")
    print("✅ All files downloaded.")
except ImportError:
    print("Running locally — files saved to current directory.")

---
## Step 12: Advanced Visualisation

| Chart | What it shows |
|---|---|
| A — Full Text Hover | Complete document in tooltip |
| B — Category Filter | Dropdown to isolate one cluster |
| C — Similarity Spotlight | Query ⭐ + dotted lines to top-5 retrieved docs |

In [ ]:
import textwrap

def wrap_text(text, width=80, max_lines=20):
    lines = []
    for para in text.strip().splitlines():
        para = para.strip()
        if not para:
            continue
        lines.extend(textwrap.wrap(para, width=width))
        if len(lines) >= max_lines:
            break
    clipped = lines[:max_lines]
    if len(lines) > max_lines:
        clipped.append("… [truncated]")
    return "<br>".join(clipped)

df["full_text_wrapped"] = [wrap_text(c.page_content) for c in chunks]

figA = px.scatter(df, x="x", y="y", color="category",
    color_discrete_map=CATEGORY_COLORS, hover_name="title",
    hover_data={"doc_id": True, "category": True, "word_count": True,
                "full_text_wrapped": True, "x": False, "y": False},
    title="📄  Chart A — Full Document Text on Hover", width=1100, height=720)
figA.update_traces(
    marker=dict(size=12, opacity=0.88, line=dict(width=0.6, color="white")),
    hoverlabel=dict(bgcolor="#1a1a2e", font_size=12, namelength=-1))
for cat, color in CATEGORY_COLORS.items():
    sub = df[df["category"] == cat]
    figA.add_annotation(x=sub["x"].mean(), y=sub["y"].mean(),
        text=f"<b>{cat.upper()}</b>", showarrow=False,
        font=dict(size=13, color=color), bgcolor="rgba(15,15,26,0.70)", borderpad=4)
dark_layout(figA).show()

In [ ]:
figB = go.Figure()
categories = sorted(df["category"].unique())

for cat in categories:
    sub = df[df["category"] == cat]
    figB.add_trace(go.Scatter(
        x=sub["x"], y=sub["y"], mode="markers", name=cat,
        marker=dict(size=11, color=CATEGORY_COLORS[cat],
                    opacity=0.88, line=dict(width=0.6, color="white")),
        hovertemplate=("<b>%{customdata[0]}</b><br>ID: %{customdata[1]}<br>"
                       "Words: %{customdata[2]}<br><i>%{customdata[3]}</i><extra></extra>"),
        customdata=sub[["title", "doc_id", "word_count", "preview"]].values,
    ))

buttons = [dict(label="All Categories", method="update",
                args=[{"visible": [True]*len(categories)},
                      {"title": "🏷️  All Categories"}])]
for i, cat in enumerate(categories):
    buttons.append(dict(label=cat.capitalize(), method="update",
        args=[{"visible": [j == i for j in range(len(categories))]},
              {"title": f"🔍  {cat.upper()} only"}]))

figB.update_layout(
    title="🏷️  Chart B — Category Filter Dropdown", title_font_size=19,
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True,
        x=0.01, xanchor="left", y=1.12, yanchor="top",
        bgcolor="#1e1e3a", bordercolor="rgba(255,255,255,0.3)",
        font=dict(color="white", size=13))],
    plot_bgcolor=DARK_BG, paper_bgcolor=DARK_BG, font_color="white",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    legend=dict(title="Category", bgcolor=LEGEND_BG, bordercolor=LEGEND_BC, borderwidth=1),
    width=1100, height=720)
figB.show()

In [ ]:
SPOTLIGHT_QUERY = "How many days of maternity leave are employees entitled to?"
TOP_K = 5

query_vec  = embedding_model.encode([SPOTLIGHT_QUERY])[0]
normed_emb = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10)
query_norm = query_vec  / (np.linalg.norm(query_vec) + 1e-10)
sims       = normed_emb @ query_norm
top_k_idx  = np.argsort(sims)[::-1][:TOP_K]
qx = df.iloc[top_k_idx]["x"].mean()
qy = df.iloc[top_k_idx]["y"].mean()

figC = go.Figure()
figC.add_trace(go.Scatter(x=df["x"], y=df["y"], mode="markers", name="All documents",
    marker=dict(size=8, color=[CATEGORY_COLORS[c] for c in df["category"]], opacity=0.20),
    hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>",
    customdata=df[["title", "preview"]].values))

top_df = df.iloc[top_k_idx].copy()
top_df["rank"] = range(1, TOP_K + 1)
top_df["sim"]  = sims[top_k_idx].round(4)
for _, row in top_df.iterrows():
    figC.add_shape(type="line", x0=qx, y0=qy, x1=row["x"], y1=row["y"],
        line=dict(color="rgba(255,255,255,0.35)", width=1.5, dash="dot"))

figC.add_trace(go.Scatter(x=top_df["x"], y=top_df["y"], mode="markers+text",
    name=f"Top {TOP_K} retrieved",
    marker=dict(size=16, color=[CATEGORY_COLORS[c] for c in top_df["category"]],
                opacity=1.0, line=dict(width=2, color="white")),
    text=[f"#{r}" for r in top_df["rank"]], textposition="top center",
    textfont=dict(size=11, color="white"),
    hovertemplate=("<b>#%{customdata[0]} — %{customdata[1]}</b><br>"
                   "Category: %{customdata[2]}<br>Similarity: %{customdata[3]}<br>"
                   "<i>%{customdata[4]}</i><extra></extra>"),
    customdata=top_df[["rank", "title", "category", "sim", "preview"]].values))

figC.add_trace(go.Scatter(x=[qx], y=[qy], mode="markers+text", name="Query",
    marker=dict(size=20, color="#FFD700", symbol="star", line=dict(width=2, color="white")),
    text=["QUERY"], textposition="bottom center", textfont=dict(size=12, color="#FFD700"),
    hovertemplate=f"<b>Query</b><br>{SPOTLIGHT_QUERY}<extra></extra>"))

figC.update_layout(
    title=f'🔍  Chart C — Similarity Spotlight: "{SPOTLIGHT_QUERY[:55]}..."',
    title_font_size=17, plot_bgcolor=DARK_BG, paper_bgcolor=DARK_BG, font_color="white",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    legend=dict(bgcolor=LEGEND_BG, bordercolor=LEGEND_BC, borderwidth=1),
    width=1100, height=740)
figC.show()

print(f"\nTop {TOP_K} nearest docs for: \"{SPOTLIGHT_QUERY}\"\n")
for rank, idx in enumerate(top_k_idx, 1):
    row = df.iloc[idx]
    print(f"  #{rank}  [{row['category']:>12}]  {row['doc_id']}  sim={sims[idx]:.4f}  {row['title']}")

---
## Step 13: RAG Query with Google Gemini

The complete RAG pipeline — retrieval from ChromaDB feeds into Gemini to generate
**grounded, cited answers** from the AcmeTech knowledge base.

```
User question
     │
     ▼
ChromaDB  ──── top-4 chunks ────►  Gemini 1.5 Flash
(vector search)                         │
                                         ▼
                              Grounded answer + sources
```

**Get your free API key:** https://aistudio.google.com/app/apikey

**Set it in Colab:**  
Left panel → 🔑 Secrets → add `GOOGLE_API_KEY`

**Or set it here (never commit to git):**
```python
import os
os.environ["GOOGLE_API_KEY"] = "your-key-here"
```

| Setting | Value | Why |
|---|---|---|
| Model | `gemini-1.5-flash` | Fast, free-tier friendly |
| `temperature` | `0.2` | Low = factual, consistent |
| `k` (retrieval) | `4` | Top 4 chunks as context |
| Prompt | Strict — context only | Prevents hallucination |

In [ ]:
# ── Colab: load key from Secrets ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    pass  # running locally — key already set via environment variable

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY:
    raise EnvironmentError(
        "GOOGLE_API_KEY not set.\n"
        "Get a free key at: https://aistudio.google.com/app/apikey"
    )
print(f"Key ends with: ...{GOOGLE_API_KEY[-4:]}")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
    max_output_tokens=1024,
)

# ── Prompt — strict: answer ONLY from context, no hallucination ───────────────
SYSTEM_PROMPT = """You are a helpful assistant for AcmeTech Solutions employees.
Answer the user's question using ONLY the context documents provided below.

Rules:
- Be concise and specific — answer the question directly.
- If the answer is a number, date, or policy detail, state it clearly.
- If the context does not contain enough information, say:
  "I couldn't find a specific answer in the AcmeTech knowledge base."
- Never make up information not present in the context.
- At the end of your answer, list the document IDs you used as sources.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
])

# ── Chain ─────────────────────────────────────────────────────────────────────
retriever        = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})
combine_chain    = create_stuff_documents_chain(llm, prompt)
rag_chain        = create_retrieval_chain(retriever, combine_chain)

print("✅ RAG chain ready  (ChromaDB → Gemini 1.5 Flash)")

In [ ]:
import textwrap

def ask(question: str) -> None:
    print("=" * 70)
    print(f"  ❓  {question}")
    print("=" * 70)

    response = rag_chain.invoke({"input": question})
    answer   = response["answer"]
    sources  = response["context"]

    # Wrap answer text for clean terminal display
    print("\n  💬  Answer:\n")
    for line in textwrap.fill(answer, width=66, subsequent_indent="  ").splitlines():
        print(f"  {line}")

    # Source attribution — deduplicated
    print(f"\n  📄  Sources ({len(sources)} chunks retrieved):\n")
    seen = set()
    for doc in sources:
        cat    = doc.metadata.get("category", "?")
        source = doc.metadata.get("source",   "?")
        if (cat, source) not in seen:
            seen.add((cat, source))
            snippet = doc.page_content.strip()[:80].replace("\n", " ")
            print(f"    [{cat:>12}]  {source}")
            print(f"               \"{snippet}...\"")
    print()

In [ ]:
# HR questions
ask("How many weeks of maternity leave does AcmeTech provide, and what is the pay structure?")
ask("What are the rules for carrying over unused annual leave?")

In [ ]:
# Finance questions
ask("How is the annual performance bonus calculated for a mid-level employee?")
ask("What is the procurement approval limit for a Director-level employee?")

In [ ]:
# Engineering questions
ask("What health checks must every Kubernetes service implement before going to production?")
ask("What is the incident severity classification for a complete platform outage?")

In [ ]:
# Support questions
ask("What should a customer do if they do not receive a password reset email?")
ask("How do I configure Single Sign-On using SAML 2.0 in AcmeTech?")

In [ ]:
# Product questions
ask("What are AcmeTech's three strategic product themes for FY2026?")
ask("How does AcmeTech calculate an OKR score and what does 0.7 mean?")

In [ ]:
# Employee questions
ask("What are Sarah Mitchell's current projects and performance rating?")
ask("Who is the Engineering Director and how many engineers do they manage?")

In [ ]:
# ── Try your own question ─────────────────────────────────────────────────────
ask("Your question here")